In [1]:
import pandas as pd
import numpy as np
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
from pathlib import Path
import os
import pynbody
import pynbody.plot.sph as sph
from IPython.display import clear_output, display
import h5py
import cupy

# fourier
from scipy.signal import stft as short_time_fft
from scipy.ndimage import gaussian_filter1d
from scipy.signal import ShortTimeFFT, find_peaks, get_window
from scipy.signal import windows

# dimensionality reduction
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

SMALLSIZE = 12
NORMALSIZE = 12
LARGESIZE = 12

params = {
    "font.family":"serif",
    "mathtext.fontset":"stix",
    "font.size": SMALLSIZE,
    "axes.titlesize" : SMALLSIZE,
    "axes.labelsize" : NORMALSIZE,
    'xtick.labelsize': SMALLSIZE,
    'ytick.labelsize': SMALLSIZE,      
    'legend.fontsize': SMALLSIZE,  
    'figure.titlesize': LARGESIZE,
    'pgf.texsystem' : "pdflatex"
}
matplotlib.rcParams.update(params)

isob_dir = "../data/IsoB_dt10_all"
snap_names = [f"GLX.000{10*(j):03.0f}" for j in range(1, 62)] + [f"GLX.0{10*j:04.0f}" for j in range(62, 101)]
iord_dir  = "../data/fil_iords.csv"

HDF5DIR = "../data/IsoB_faceon.h5"
h5keys = [f"GLX{10*(x+1):04.0f}" for x in range(100)]


In [1]:
with h5py.File('../data/data_cube.h5', 'r') as f:
    data_c = f['coords_data'][:]

print(data_c.shape)

NameError: name 'h5py' is not defined

In [12]:
df = pd.read_hdf(f"../data/fIsoB_faceon.h5", key="GLX1000")
df_sort = df.set_index('iord')
#print(df_sort)

full_iords = df['iord']

print(full_iords)

# 1007166982, 1007288787, 1010666409 

0          1006000000
1          1006000001
2          1006000002
3          1006000003
4          1006000004
              ...    
4666405    1010666405
4666406    1010666406
4666407    1010666407
4666408    1010666408
4666409    1010666409
Name: iord, Length: 4590904, dtype: int64


In [2]:
all_iords = set()
for key in h5keys:
    df_iords = pd.read_hdf(f"../data/fIsoB_faceon.h5", key=key)
    all_iords.update(df_iords['iord'].values)

target_iords = np.array(sorted(list(all_iords)))
print(f"Total unique objects: {len(target_iords)}")

Total unique objects: 4590904


In [3]:
np.savetxt("../data/all_iords.csv", target_iords, delimiter=",")

In [16]:
h5keys = [f"GLX{10*(x+1):04.0f}" for x in range(100)]


data_cube = np.full((3, 100, len(target_iords)), np.nan)
print(data_cube.shape)


for i, key in enumerate(h5keys):
    df = pd.read_hdf(f"../data/fIsoB_faceon.h5", key=key)
    df_sort = df.set_index('iord').reindex(target_iords)

    data_cube[0, i, :] = df_sort['x'].values
    data_cube[1, i, :] = df_sort['y'].values
    data_cube[2, i, :] = df_sort['z'].values

    print(i)

(3, 100, 4590904)
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99


In [18]:
print(data_cube.shape)

(3, 100, 4590904)


In [ ]:
with h5py.File("../data/all_stars_dc.h5", 'w') as f:
    dset = f.create_dataset('coords_data', data=data_cube, compression='gzip', chunks=True)
    dset.attrs['description'] = "Data cube shape (3, 100, 189513) for (Axis, Timestep, Iord)"

: 

In [4]:
with h5py.File('../data/all_stars_dc.h5', 'r') as f:
    data_c = f['coords_data'][:]

In [5]:
print(data_c.shape)

(3, 100, 4590904)
